In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container {width:100% !important; }</style>"))

import glob
import pandas as pd
import numpy as np
import plotly.graph_objects as go
pd.options.mode.chained_assignment = None

import datetime as dt

from pathlib import Path
import re

PLOT_DIR = Path("plots")  # scripts/plots/ when the notebook runs from scripts/
PLOT_DIR.mkdir(exist_ok=True)

# Backup old plots
old_plots = glob.glob(str(PLOT_DIR / "*.html"))
if len(old_plots) > 0:
    BAK_DIR = PLOT_DIR / (dt.datetime.now().strftime("%Y%m%d_%H%M%S"))
    BAK_DIR.mkdir(exist_ok=True)
    for f in old_plots:
        Path(f).rename(BAK_DIR / Path(f).name)

def save_plot(fig, name):
    """Save a Plotly figure as HTML under scripts/plots/ (plotly.js via CDN to keep files small)."""
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")
    path = PLOT_DIR / f"{safe}.html"
    fig.write_html(str(path), include_plotlyjs="cdn")
    print(f"saved {path}")
    return path

%autosave 30

In [ ]:
CPU_FREQ=2.60

rdtsc_df_dict = []
ttt_df_dict = []
for filename in glob.glob("../logs/exchange*.log") + glob.glob("../logs/*_1.log"):
    print('processing {}'.format(filename))
    for line in open(filename):
        tokens = line.strip().split(' ')
        if len(tokens) != 6:
            continue
        if "[DEBUG]" not in tokens[2]:
            continue

        try:
            time = tokens[1][:-1]
            tag = tokens[4]
            latency = float(tokens[5])
            latency_rdtsc = latency / CPU_FREQ
            time_datetime = pd.to_datetime(time, format='%H:%M:%S.%f')
        except:
            continue

        if ' RDTSC ' in line:
            if tokens[3] != 'RDTSC':
                continue

            rdtsc_df_dict.append({'timestamp':time, 'tag':tag, 'latency':latency_rdtsc})
        elif ' TTT ' in line:
            if tokens[3] != 'TTT':
                continue

            ttt_df_dict.append({'timestamp':time, 'tag':tag, 'latency':latency})

    print(len(rdtsc_df_dict), len(ttt_df_dict))
        
rdtsc_df = pd.DataFrame.from_dict(rdtsc_df_dict)
rdtsc_df = rdtsc_df.drop_duplicates().sort_values(by='timestamp')
rdtsc_df['timestamp'] = pd.to_datetime(rdtsc_df['timestamp'], format='%H:%M:%S.%f')

ttt_df = pd.DataFrame.from_dict(ttt_df_dict)
ttt_df = ttt_df.drop_duplicates().sort_values(by='timestamp')
ttt_df['timestamp'] = pd.to_datetime(ttt_df['timestamp'], format='%H:%M:%S.%f')

In [ ]:
# Per-component latency (RDTSC, converted to ns) with rolling mean / p50 / p99 / p99.9
ROLLING_FRAC = 100      # rolling window = len / ROLLING_FRAC
MIN_WINDOW = 10        # p99 needs ~100+ pts, p99.9 ideally ~1000+; 200 = compromise, p99.9 ~= near-max

summary_rows = []
for tag in sorted(rdtsc_df['tag'].unique()):
    t_df = rdtsc_df[rdtsc_df['tag'] == tag].copy()
    # drop non-positive artefacts only; do NOT clip tails (that would bias p99/p99.9)
    t_df = t_df[t_df['latency'] > 0]
    if len(t_df) == 0:
        continue
    lat = t_df['latency'].astype(float)
    summary_rows.append({
        'tag': tag,
        'n': len(t_df),
        'mean': lat.mean(),
        'p50': lat.quantile(0.5),
        'p99': lat.quantile(0.99),
        'p99.9': lat.quantile(0.999),
        'max': lat.max(),
    })

summary = pd.DataFrame(summary_rows).set_index('tag')
unit = 'ns'
if (summary['p50'] >= 1000).all():
    unit = 'us'
    summary[['mean', 'p50', 'p99', 'p99.9', 'max']] /= 1000
print(f'RDTSC summary (in {unit}):')
display(summary.style.format('{:.3f}', subset=['mean', 'p50', 'p99', 'p99.9', 'max']))

for tag in summary.index:
    t_df = rdtsc_df[rdtsc_df['tag'] == tag].copy()
    t_df = t_df[t_df['latency'] > 0].sort_values('timestamp')
    lat_raw = t_df['latency'].astype(float)
    row = summary.loc[tag]
    print(f"{tag}: n={int(row['n'])} mean={row['mean']:.3f}{unit} "
          f"p50={row['p50']:.3f}{unit} p99={row['p99']:.3f}{unit} "
          f"p99.9={row['p99.9']:.3f}{unit} max={row['max']:.3f}{unit}")

    plot_unit, div = 'nanoseconds', 1.0
    if lat_raw.median() >= 1000:
        plot_unit, div = 'microseconds', 1000.0
    t_df['lat_plot'] = lat_raw / div

    window = max(MIN_WINDOW, int(len(t_df) / ROLLING_FRAC))
    roll = t_df['lat_plot'].rolling(window, min_periods=window, center=True)  # require full window: partial edge windows make p99/p99.9 ~= max)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['lat_plot'], name=tag + ' raw',
                             mode='markers', marker=dict(size=3, opacity=0.25)))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=roll.mean(), name='rolling mean'))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=roll.quantile(0.5), name='rolling p50'))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=roll.quantile(0.99), name='rolling p99'))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=roll.quantile(0.999), name='rolling p99.9'))
    fig.update_layout(title=f'performance {tag} ({plot_unit}, window={window})',
                      height=600, width=1000, hovermode='x',
                      legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
                      yaxis_title=plot_unit)
    save_plot(fig, f"rdtsc_{tag}")
    fig.show()

In [ ]:
HOPS = [
    ['T1_OrderServer_TCP_read', 'T2_OrderServer_LFQueue_write'],
    ['T2_OrderServer_LFQueue_write', 'T3_MatchingEngine_LFQueue_read'],
    ['T3_MatchingEngine_LFQueue_read', 'T4_MatchingEngine_LFQueue_write'], ['T3_MatchingEngine_LFQueue_read', 'T4t_MatchingEngine_LFQueue_write'],
    ['T4_MatchingEngine_LFQueue_write', 'T5_MarketDataPublisher_LFQueue_read'], ['T4t_MatchingEngine_LFQueue_write', 'T5t_OrderServer_LFQueue_read'],
    ['T5_MarketDataPublisher_LFQueue_read', 'T6_MarketDataPublisher_UDP_write'], ['T5t_OrderServer_LFQueue_read', 'T6t_OrderServer_TCP_write'],
    ['T7_MarketDataConsumer_UDP_read', 'T8_MarketDataConsumer_LFQueue_write'], ['T7t_OrderGateway_TCP_read', 'T8t_OrderGateway_LFQueue_write'],
    ['T8_MarketDataConsumer_LFQueue_write', 'T9_TradeEngine_LFQueue_read'], ['T8t_OrderGateway_LFQueue_write', 'T9t_TradeEngine_LFQueue_read'],
    ['T9_TradeEngine_LFQueue_read', 'T10_TradeEngine_LFQueue_write'], ['T9t_TradeEngine_LFQueue_read', 'T10_TradeEngine_LFQueue_write'],
    ['T10_TradeEngine_LFQueue_write', 'T11_OrderGateway_LFQueue_read'],
    ['T11_OrderGateway_LFQueue_read', 'T12_OrderGateway_TCP_write'],
    # exchange <-> client
    ['T12_OrderGateway_TCP_write', 'T1_OrderServer_TCP_read'],
    ['T6_MarketDataPublisher_UDP_write', 'T7_MarketDataConsumer_UDP_read'], ['T6t_OrderServer_TCP_write', 'T7t_OrderGateway_TCP_read'],
]

In [ ]:
# Hop-to-hop latency (TTT differences) with rolling mean / p50 / p99 / p99.9
ROLLING_FRAC = 100
MIN_WINDOW = 200  # p99 needs ~100+ pts, p99.9 ideally ~1000+; 200 = compromise, p99.9 ~= near-max

hop_summary = []
for tags in HOPS:
    tag_p, tag_n = tags
    n_p = int((ttt_df['tag'] == tag_p).sum())
    n_n = int((ttt_df['tag'] == tag_n).sum())
    print(f'{tag_p} => {n_p}. {tag_n} => {n_n}.')

    t_df = ttt_df[(ttt_df['tag'] == tag_n) | (ttt_df['tag'] == tag_p)].copy()
    t_df = t_df.sort_values('timestamp')
    t_df['latency_diff'] = t_df['latency'].diff()
    t_df = t_df[(t_df['latency_diff'] > 0) & (t_df.tag == tag_n)]
    if len(t_df) == 0:
        print(f'{tag_n} has 0 observations, skipping')
        continue

    lat = t_df['latency_diff'].astype(float)
    hop_summary.append({
        'hop': f'{tag_p} -> {tag_n}',
        'n': len(t_df),
        'mean': lat.mean(),
        'p50': lat.quantile(0.5),
        'p99': lat.quantile(0.99),
        'p99.9': lat.quantile(0.999),
        'max': lat.max(),
    })

    window = max(MIN_WINDOW, int(len(t_df) / ROLLING_FRAC))
    unit, div = 'nanoseconds', 1.0
    if lat.median() >= 1_000_000:
        unit, div = 'milliseconds', 1_000_000.0
    elif lat.median() >= 1000:
        unit, div = 'microseconds', 1000.0
    t_df['diff_plot'] = lat / div

    roll = t_df['diff_plot'].rolling(window, min_periods=window, center=True)  # require full window: partial edge windows make p99/p99.9 ~= max)
    tag_name = f'{tag_p} -> {tag_n}'
    row = hop_summary[-1]
    print('{} has {} observations mean {:.1f} p50 {:.1f} p99 {:.1f} p99.9 {:.1f} ns (raw)'.format(
        tag_n, len(t_df), row['mean'], row['p50'], row['p99'], row['p99.9']))

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=t_df['diff_plot'], name=tag_name + ' raw',
                             mode='markers', marker=dict(size=3, opacity=0.25)))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=roll.mean(), name=tag_name + ' mean'))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=roll.quantile(0.5), name=tag_name + ' p50'))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=roll.quantile(0.99), name=tag_name + ' p99'))
    fig.add_trace(go.Scatter(x=t_df['timestamp'], y=roll.quantile(0.999), name=tag_name + ' p99.9'))
    fig.update_layout(title=f'performance {tag_name} ({unit}, window={window})',
                      height=600, width=1000, hovermode='x',
                      legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
                      yaxis_title=unit)
    save_plot(fig, f"hop_{tag_p}_to_{tag_n}")
    fig.show()

hop_summary_df = pd.DataFrame(hop_summary).set_index('hop')
display(hop_summary_df.style.format('{:.1f}', subset=['mean', 'p50', 'p99', 'p99.9', 'max']).format('{:.0f}', subset=['n']))

In [ ]:
# Overview: p50 / p99 / p99.9 across all RDTSC components and all hops (log y-axis)
def _plot_overview(df_ns, title, fname):
    fig = go.Figure()
    for col in ['p50', 'p99', 'p99.9']:
        fig.add_trace(go.Bar(x=df_ns.index, y=df_ns[col], name=col))
    fig.update_layout(title=title, height=500, width=1100, barmode='group',
                      yaxis_type='log', yaxis_title='ns (log scale)', xaxis_tickangle=-30)
    save_plot(fig, fname)
    fig.show()

if 'summary' in globals() and len(summary) > 0:
    s_ns = summary.copy()
    if unit == 'us':  # summary table was scaled to us; convert back for common ns scale
        s_ns[['mean', 'p50', 'p99', 'p99.9', 'max']] *= 1000
    _plot_overview(s_ns, 'RDTSC p50 / p99 / p99.9 per component (ns, log scale)', 'overview_rdtsc_percentiles')

if 'hop_summary_df' in globals() and len(hop_summary_df) > 0:
    _plot_overview(hop_summary_df, 'Hop latency p50 / p99 / p99.9 per link (ns, log scale)', 'overview_hop_percentiles')